In [1]:
try:
    import xgboost
    print("xgboost available:", xgboost.__version__)
except ImportError:
    print("xgboost NOT available")

xgboost available: 3.2.0


In [2]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import pandas as pd
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


train = pd.read_csv('../train_cleaned.csv')
test = pd.read_csv('../test_cleaned.csv')
feature_cols = [
    "Applied_Voltage_kV", "Load_Current_A", "Ambient_Temperature_C", "Test_Duration_min",
    "Sensor_S1", "Sensor_S2", "Sensor_S3",
    "S1_missing", "S2_missing", "S3_missing", "S4_missing",
    "is_duplicate_input"]


X = train[feature_cols]
y_class = train["Validity_Label_enc"]
y_reg = train["Reference_Parameter"]


xgb_f1_scores = []
xgb_auc_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

    # scale_pos_weight handles imbalance for XGBoost (its equivalent of class_weight="balanced")
    scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()

    clf_xgb = xgb.XGBClassifier(
        n_estimators=300,
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss"
    )
    clf_xgb.fit(X_tr, y_tr)

    preds = clf_xgb.predict(X_vl)
    probs = clf_xgb.predict_proba(X_vl)[:, 1]

    f1 = f1_score(y_vl, preds)
    auc = roc_auc_score(y_vl, probs)
    xgb_f1_scores.append(f1)
    xgb_auc_scores.append(auc)

    print(f"Fold {fold}: F1={f1:.4f}, AUC={auc:.4f}")

print(f"\nMean F1: {np.mean(xgb_f1_scores):.4f} (+/- {np.std(xgb_f1_scores):.4f})")
print(f"Mean AUC: {np.mean(xgb_auc_scores):.4f} (+/- {np.std(xgb_auc_scores):.4f})")

Fold 0: F1=0.8696, AUC=0.9965
Fold 1: F1=0.9200, AUC=0.9985
Fold 2: F1=0.9200, AUC=0.9902
Fold 3: F1=0.8750, AUC=0.9966
Fold 4: F1=0.8980, AUC=0.9884

Mean F1: 0.8965 (+/- 0.0214)
Mean AUC: 0.9940 (+/- 0.0040)


In [4]:
from sklearn.metrics import precision_recall_curve

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

    scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()

    clf_xgb = xgb.XGBClassifier(
        n_estimators=300, random_state=42,
        scale_pos_weight=scale_pos_weight, eval_metric="logloss"
    )
    clf_xgb.fit(X_tr, y_tr)
    probs = clf_xgb.predict_proba(X_vl)[:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_vl, probs)
    f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
    best_idx = f1s.argmax()
    print(f"Fold {fold}: best threshold={thresholds[best_idx]:.3f}, best F1={f1s[best_idx]:.4f} (default 0.5 F1 was {xgb_f1_scores[fold]:.4f})")

Fold 0: best threshold=0.377, best F1=0.9388 (default 0.5 F1 was 0.8696)
Fold 1: best threshold=0.118, best F1=0.9474 (default 0.5 F1 was 0.9200)
Fold 2: best threshold=0.502, best F1=0.9200 (default 0.5 F1 was 0.9200)
Fold 3: best threshold=0.080, best F1=0.9630 (default 0.5 F1 was 0.8750)
Fold 4: best threshold=0.138, best F1=0.9811 (default 0.5 F1 was 0.8980)


In [5]:
print("RF tuned F1 std:", np.std([0.9231, 0.9630, 0.9474, 0.9643, 0.9643]))  # your actual RF Step 2f numbers
print("XGB tuned F1 std:", np.std([0.9388, 0.9474, 0.9200, 0.9630, 0.9811]))

RF tuned F1 std: 0.015994173939281764
XGB tuned F1 std: 0.02082494657856291


##  This to test for the Regression


In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_reg_rmse = []
xgb_reg_mae = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_reg.iloc[train_idx], y_reg.iloc[val_idx]

    reg_xgb = xgb.XGBRegressor(
        n_estimators=300,
        random_state=42
    )
    reg_xgb.fit(X_tr, y_tr)

    preds = reg_xgb.predict(X_vl)

    rmse = np.sqrt(mean_squared_error(y_vl, preds))
    mae = mean_absolute_error(y_vl, preds)
    xgb_reg_rmse.append(rmse)
    xgb_reg_mae.append(mae)

    print(f"Fold {fold}: RMSE={rmse:.4f}, MAE={mae:.4f}")

print(f"\nMean RMSE: {np.mean(xgb_reg_rmse):.4f} (+/- {np.std(xgb_reg_rmse):.4f})")
print(f"Mean MAE: {np.mean(xgb_reg_mae):.4f} (+/- {np.std(xgb_reg_mae):.4f})")

Fold 0: RMSE=1.2108, MAE=0.7499
Fold 1: RMSE=2.8470, MAE=1.0314
Fold 2: RMSE=1.9263, MAE=0.8478
Fold 3: RMSE=3.2231, MAE=1.1069
Fold 4: RMSE=2.8894, MAE=1.1816

Mean RMSE: 2.4193 (+/- 0.7421)
Mean MAE: 0.9835 (+/- 0.1611)
